In [1]:
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder \
    .appName("Filter Examples") \
    .master("local[2]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/24 09:10:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
! ls -l /opt/examples/datasets | grep Mall_Customers

-rw-r--r-- 1 ssh_train users     4365 Oct 24 08:14 Mall_Customers.csv


In [4]:
df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ",") \
    .load("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()
df.limit(5).toPandas()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)



,CustomerID,Gender,Age,AnnualIncome,SpendingScore
0,1,Male,19,15000,39
1,2,Male,21,15000,81
2,3,Female,20,16000,6
3,4,Female,23,16000,77
4,5,Female,31,17000,40


In [6]:
df.where("Gender = 'Male'").show(5)

+----------+------+---+------------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|
+----------+------+---+------------+-------------+
|         1|  Male| 19|       15000|           39|
|         2|  Male| 21|       15000|           81|
|         9|  Male| 64|       19000|            3|
|        11|  Male| 67|       19000|           14|
|        15|  Male| 37|       20000|           13|
+----------+------+---+------------+-------------+
only showing top 5 rows


In [7]:
df.filter( (F.col("Gender") == "Female") & (F.col("Age") < 30) ).show(5)

+----------+------+---+------------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|
+----------+------+---+------------+-------------+
|         3|Female| 20|       16000|            6|
|         4|Female| 23|       16000|           77|
|         6|Female| 22|       17000|           76|
|         8|Female| 23|       18000|           94|
|        14|Female| 24|       20000|           77|
+----------+------+---+------------+-------------+
only showing top 5 rows


In [8]:
df.filter("AnnualIncome BETWEEN 40000 AND 60000").show(5)

+----------+------+---+------------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|
+----------+------+---+------------+-------------+
|        47|Female| 50|       40000|           55|
|        48|Female| 27|       40000|           47|
|        49|Female| 29|       40000|           42|
|        50|Female| 31|       40000|           42|
|        51|Female| 49|       42000|           52|
+----------+------+---+------------+-------------+
only showing top 5 rows


In [9]:
df.columns

['CustomerID', 'Gender', 'Age', 'AnnualIncome', 'SpendingScore']

In [10]:
df.dtypes

[('CustomerID', 'int'),
 ('Gender', 'string'),
 ('Age', 'int'),
 ('AnnualIncome', 'int'),
 ('SpendingScore', 'int')]

In [15]:
df2 = df
for col in df.dtypes:
    
    if col[1] == 'string':
        df2 = df2.withColumn(col[0], F.trim(col[0]))
        print(col[0])

Gender


In [16]:
df2.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)



In [17]:
df = spark.read \
.option("header",True) \
.option("inferSchema", True) \
.csv("file:///opt/examples/datasets/Hotel_Reviews.csv.gz")

In [20]:
def upper_case(x):
    return x.upper()

In [21]:
upper_case("Ali gel.")

'ALI GEL.'

In [22]:
from pyspark.sql.types import StringType

upper_case_udf = F.udf(upper_case, StringType())

In [24]:
df.select("Reviewer_Nationality",upper_case_udf("Reviewer_Nationality")).show(5)

+--------------------+--------------------------------+
|Reviewer_Nationality|upper_case(Reviewer_Nationality)|
+--------------------+--------------------------------+
|             Russia |                         RUSSIA |
|            Ireland |                        IRELAND |
|          Australia |                      AUSTRALIA |
|     United Kingdom |                 UNITED KINGDOM |
|        New Zealand |                    NEW ZEALAND |
+--------------------+--------------------------------+
only showing top 5 rows


In [28]:
from pyspark.sql.types import *

In [30]:
@F.udf(FloatType())
def decorator_multiply_by_three(number):
    return number * 3
 
df.withColumn(
    "decorator_times_three", decorator_multiply_by_three("Average_Score")
).limit(3).toPandas()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng,decorator_times_three
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.3605759,4.9159683,23.1
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.3605759,4.9159683,23.1
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.3605759,4.9159683,23.1


26/10/24 09:32:58 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-09998cba-68f2-4f9a-918f-1034be67de35/pyspark-dad72e3f-cb0a-44f3-91df-f368ebfca585. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-09998cba-68f2-4f9a-918f-1034be67de35/pyspark-dad72e3f-cb0a-44f3-91df-f368ebfca585
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfu